# RAG: Ollama + pgvector

**Dando Memoria a la IA con Embeddings & RAG** · Henry

---

## ¿Qué vamos a construir?

Un sistema de **preguntas y respuestas** que:
1. Toma un documento PDF y lo divide en fragmentos (chunks).
2. Convierte cada fragmento en un vector numérico (embedding).
3. Guarda esos vectores en **PostgreSQL + pgvector**.
4. Cuando llega una pregunta, busca los fragmentos más relevantes y se los pasa a un LLM para que responda.

```
INDEXING:  PDF  →  Chunking (tiktoken)  →  nomic-embed-text  →  pgvector
QUERY:     Pregunta  →  nomic-embed-text  →  pgvector (<=>)  →  top-K chunks  →  llama3.1:8b  →  Respuesta
```

---

## Stack tecnológico

| Componente | Herramienta | Descripción |
|---|---|---|
| Embedding | `nomic-embed-text` via Ollama | Vectoriza texto, 768 dims, 100% local |
| Generación | `llama3.1:8b` via Ollama | Responde con contexto aumentado |
| Vector store | PostgreSQL + pgvector | Índice HNSW, búsqueda por similitud |

---

## Pre-requisitos (correr ANTES de la clase)

```bash
# 1. Levantar PostgreSQL con pgvector
docker run -d --name pgvector-rag \
  -e POSTGRES_PASSWORD=postgres \
  -e POSTGRES_DB=rag_demo \
  -p 5432:5432 \
  pgvector/pgvector:pg16

# 2. Descargar modelos en Ollama
ollama pull nomic-embed-text
ollama pull llama3.1:8b
```


---
## Celda 1 — Instalación de dependencias

Instala todas las librerías necesarias. Solo hay que correr esto una vez.


In [1]:
%pip install ollama psycopg2-binary pgvector tiktoken pymupdf python-dotenv -q



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---
## Celda 2 — Imports, configuración y funciones base

Toda la configuración y las funciones auxiliares en **una sola celda**.
Así el notebook es ejecutable de arriba a abajo sin importar el orden.

Crea un archivo `.env` en la misma carpeta con:
```
DB_HOST=localhost
DB_PORT=5432
DB_NAME=rag_demo
DB_USER=postgres
DB_PASSWORD=postgres
```


In [2]:
import os
import json
import psycopg2
import tiktoken
import ollama
import fitz  # pymupdf — extrae texto de PDFs

from pathlib import Path
from dotenv import load_dotenv
from psycopg2.extras import execute_values

load_dotenv()  # lee variables del archivo .env

# ── Conexión a PostgreSQL ─────────────────────────────────────
# Con Docker Compose: DB_HOST=db  (nombre del servicio)
# En local sin Docker: DB_HOST=localhost
# El .env.example tiene todos los valores por defecto.
DB_CONFIG = {
    "host":     os.getenv("DB_HOST",     "localhost"),
    "port":     int(os.getenv("DB_PORT", 5432)),
    "dbname":   os.getenv("DB_NAME",     "rag_demo"),
    "user":     os.getenv("DB_USER",     "postgres"),
    "password": os.getenv("DB_PASSWORD", "postgres"),
}

# ── Modelos Ollama ────────────────────────────────────────────
EMBED_MODEL = "nomic-embed-text"  # vectoriza texto → 768 dimensiones
CHAT_MODEL  = "llama3.2:1b"      # genera la respuesta final

# ── Parámetros de chunking ────────────────────────────────────
CHUNK_SIZE    = 400  # tokens por fragmento
CHUNK_OVERLAP = 50   # tokens compartidos entre fragmentos consecutivos
TOP_K         = 5    # chunks a recuperar por consulta


def get_conn():
    """Retorna una conexión fresca a PostgreSQL."""
    return psycopg2.connect(**DB_CONFIG)


def embed(text: str) -> list:
    """Convierte un texto en un vector de 768 números con nomic-embed-text."""
    return ollama.embed(model=EMBED_MODEL, input=text)["embeddings"][0]


def embed_batch(texts: list) -> list:
    """Vectoriza una lista de textos en una sola llamada (más eficiente que un loop)."""
    return ollama.embed(model=EMBED_MODEL, input=texts)["embeddings"]


def chunk_text(text: str) -> list:
    """
    Divide el texto en fragmentos de CHUNK_SIZE tokens.
    Cada fragmento comparte CHUNK_OVERLAP tokens con el siguiente
    para no perder contexto en los bordes.
    """
    enc    = tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    chunks = []

    start = 0
    while start < len(tokens):
        end = min(start + CHUNK_SIZE, len(tokens))
        chunks.append(enc.decode(tokens[start:end]).strip())
        start += CHUNK_SIZE - CHUNK_OVERLAP  # avanzar dejando overlap

    return chunks


# Verificar que Ollama responde antes de continuar
v = embed("test")
print(f"Ollama embed OK — dimensiones: {len(v)}")
print(f"Conectando a PostgreSQL en {DB_CONFIG['host']}:{DB_CONFIG['port']}...")
with get_conn() as conn:
    print("PostgreSQL OK ✓")
print("Configuración lista ✓")


Ollama embed OK — dimensiones: 768
Conectando a PostgreSQL en db:5432...
PostgreSQL OK ✓
Configuración lista ✓


---
## Celda 3 — Setup de la base de datos

Habilitamos la extensión `vector` de pgvector y creamos la tabla `document_chunks`.

El índice **HNSW** permite búsqueda aproximada de alta velocidad.
A diferencia de IVFFlat, no requiere ejecutar `VACUUM` periódicamente.

```
document_chunks
  id         SERIAL       — primary key
  source     TEXT         — nombre del archivo
  chunk_idx  INTEGER      — posición dentro del documento
  content    TEXT         — texto del chunk
  embedding  vector(768)  — vector nomic-embed-text
```


In [3]:
def setup_db():
    """
    Crea la extensión vector, la tabla document_chunks y el índice HNSW.
    NOTA: hace DROP de la tabla si ya existe para asegurar el schema correcto.
    """
    with get_conn() as conn:
        with conn.cursor() as cur:

            # Habilitar pgvector (solo la primera vez por base de datos)
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

            # Recrear la tabla limpia — elimina la versión anterior si existe
            # (necesario si venías de una versión con chunk_type)
            cur.execute("DROP TABLE IF EXISTS document_chunks CASCADE;")

            cur.execute("""
                CREATE TABLE document_chunks (
                    id        SERIAL  PRIMARY KEY,
                    source    TEXT    NOT NULL,
                    chunk_idx INTEGER NOT NULL,
                    content   TEXT    NOT NULL,
                    embedding vector(768)
                );
            """)

            # HNSW para cosine similarity
            cur.execute("""
                CREATE INDEX chunks_hnsw_idx
                    ON document_chunks
                    USING hnsw (embedding vector_cosine_ops);
            """)

        conn.commit()
    print("DB lista — tabla document_chunks recreada + índice HNSW ✓")


setup_db()


DB lista — tabla document_chunks recreada + índice HNSW ✓


---
## Celda 4 — Indexing de texto (PDF)

Pipeline completo para indexar un documento:

1. **Extracción**: PyMuPDF lee el PDF página a página y concatena el texto.
2. **Chunking**: dividimos en fragmentos de 400 tokens con 50 de overlap.
3. **Embedding en batch**: una sola llamada a Ollama para todos los chunks.
4. **Inserción masiva**: `execute_values` inserta todo en una transacción.

La operación es **idempotente**: si el archivo ya fue indexado, primero borra
los registros anteriores antes de reinsertar.


In [4]:
def index_pdf(pdf_path: str):
    """
    Indexa un PDF en pgvector:
    extrae texto → chunking → embed batch → inserción masiva.
    """
    source = Path(pdf_path).name
    print(f"Indexando: {source}")

    # Extraer texto de todas las páginas
    doc  = fitz.open(pdf_path)
    text = "\n".join(p.get_text() for p in doc)

    chunks = chunk_text(text)
    print(f"  → {len(chunks)} chunks generados")

    # Embed en batch: más rápido que llamar embed() en un loop
    embeddings = embed_batch(chunks)
    print(f"  → {len(embeddings)} embeddings generados")

    # Preparar filas para inserción masiva
    rows = [
        (source, idx, chunk, json.dumps(emb))
        for idx, (chunk, emb) in enumerate(zip(chunks, embeddings))
    ]

    with get_conn() as conn:
        with conn.cursor() as cur:
            # Borrar indexación previa del mismo archivo (idempotente)
            cur.execute("DELETE FROM document_chunks WHERE source=%s;", (source,))
            execute_values(
                cur,
                "INSERT INTO document_chunks (source, chunk_idx, content, embedding) VALUES %s",
                rows,
                template="(%s, %s, %s, %s::vector)",
            )
        conn.commit()

    print(f"  → {len(rows)} chunks guardados en pgvector ✓")


index_pdf("techpulse_ia_2025.pdf")


Indexando: techpulse_ia_2025.pdf
  → 6 chunks generados
  → 6 embeddings generados
  → 6 chunks guardados en pgvector ✓


---
## Celda 5 — Retrieval: búsqueda por similitud en pgvector

El operador `<=>` de pgvector calcula la **distancia coseno** entre dos vectores.
Menor distancia = mayor similitud semántica.

Usamos `1 - (embedding <=> query_vec)` para convertirlo en un **score de similitud**
entre 0 y 1, más intuitivo para mostrar en los resultados.


In [5]:
def retrieve(query: str, top_k: int = TOP_K) -> list:
    """
    Vectoriza la pregunta y recupera los top_k chunks más similares de pgvector.
    Retorna una lista de dicts con source, chunk_idx, content y similarity.
    """
    q_vec = json.dumps(embed(query))

    sql = """
        SELECT
            source,
            chunk_idx,
            content,
            1 - (embedding <=> %s::vector) AS similarity
        FROM document_chunks
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """

    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, [q_vec, q_vec, top_k])
            rows = cur.fetchall()

    return [
        {
            "source":    r[0],
            "chunk_idx": r[1],
            "content":   r[2],
            "similarity": round(float(r[3]), 4),
        }
        for r in rows
    ]


# Demo: ver qué chunks recupera una pregunta
query = "¿Qué herramientas usan los desarrolladores en LATAM?"
hits  = retrieve(query)

print(f"Query: {query}\n")
for h in hits:
    print(f"  [{h['similarity']:.4f}] {h['source']}")
    print(f"  {h['content'][:120].replace(chr(10), ' ')}...\n")


Query: ¿Qué herramientas usan los desarrolladores en LATAM?

  [0.7034] techpulse_ia_2025.pdf
  TechPulse Research  |  Reporte Q2 2025 Estado del Arte: IA Generativa en Ingenieria de Software 2025 TechPulse Research ...

  [0.6557] techpulse_ia_2025.pdf
  % interanual) - Cursor / Windsurf IDEs IA      -  18% Casos de uso principales - Autocompletado y generacion de codigo b...

  [0.6442] techpulse_ia_2025.pdf
  izan o eliminan facilmente. - Privacidad: con Ollama + nomic-embed, los datos nunca salen de la infraestructura. Stack t...

  [0.6318] techpulse_ia_2025.pdf
  ecosistema (pgvector, Ollama, LangChain) reduce la barrera de entrada a dias. Los equipos reportan ROI positivo en menos...

  [0.6063] techpulse_ia_2025.pdf
  ario, soporte para Llama 3, Mistral, Phi-3 y nomic-embed-text. Adopcion critica en fintech, salud y gobierno por privaci...



---
## Celda 6 — Generation: prompt aumentado + llama3.1:8b

Con los chunks recuperados construimos un prompt que incluye el contexto
etiquetado por fuente. Le indicamos al LLM que responda **solo con lo que
está en el contexto** y que cite la fuente cuando sea relevante.

`temperature=0.2` hace las respuestas precisas y consistentes.
Una temperatura más alta produciría respuestas más creativas pero menos
fieles al contexto recuperado.


In [6]:
SYSTEM_PROMPT = """Eres un asistente especializado en IA y desarrollo de software.
Responde ÚNICAMENTE usando los fragmentos de contexto proporcionados.
Cada fragmento indica su fuente.
Si la respuesta no está en el contexto, indícalo explícitamente.
Cita la fuente cuando sea relevante."""


def build_context(chunks: list) -> str:
    """Concatena los chunks en un bloque de contexto etiquetado por fuente."""
    parts = [
        f"[{i} | Fuente: {c['source']} | Similitud: {c['similarity']}]\n{c['content']}"
        for i, c in enumerate(chunks, 1)
    ]
    return "\n\n---\n\n".join(parts)


def ask(question: str, top_k: int = TOP_K) -> str:
    """
    Pipeline RAG completo (Retrieval → Augment → Generate):
      1. Vectoriza la pregunta con nomic-embed-text
      2. Recupera top_k chunks relevantes de pgvector
      3. Construye el prompt aumentado con el contexto
      4. llama3.1:8b genera la respuesta localmente
    """
    chunks   = retrieve(question, top_k)
    context  = build_context(chunks)
    user_msg = f"Contexto:\n{context}\n\nPregunta: {question}"

    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        options={"temperature": 0.2},  # baja temperatura = respuestas precisas
    )
    return response["message"]["content"]


print("Pipeline RAG listo ✓")


Pipeline RAG listo ✓


---
## Celda 7 — Demo: preguntas sobre el reporte TechPulse

Corremos el pipeline completo con cinco preguntas.
El sistema recupera los chunks más relevantes y genera respuestas
contextualizadas con **llama3.1:8b corriendo localmente**.


In [7]:
preguntas = [
    "¿Qué porcentaje de equipos usa pgvector como vector store?",
    "¿Por qué RAG es preferido sobre fine-tuning para datos que cambian frecuentemente?",
    "¿Qué ventajas tiene usar Ollama con modelos locales?",
    "¿Cuántas horas semanales ahorra RAG por desarrollador?",
    "¿Qué es Agentic RAG y por qué es una tendencia emergente?",
]

for pregunta in preguntas:
    print(f"\n{'='*65}")
    print(f"Pregunta: {pregunta}")
    print(f"{'='*65}")
    print(ask(pregunta))



Pregunta: ¿Qué porcentaje de equipos usa pgvector como vector store?
Según el texto proporcionado, el porcentaje de equipos que usan pgvector como vector store es del 38%. Esto se refleja en la siguiente cita:

"Stack tecnologico mas comun para RAG (2025)
- Vector stores: pgvector/PostgreSQL 38% | Qdrant 22% | ChromaDB 19% | Pinecone 15%"

Pregunta: ¿Por qué RAG es preferido sobre fine-tuning para datos que cambian frecuentemente?
No se proporcionó una pregunta específica en el contexto proporcionado. Sin embargo, puedo ofrecerte una respuesta general basada en los fragmentos de contexto proporcionados.

Según la información proporcionada, RAG (Retrieval-Augmented Generation) es preferido sobre fine-tuning para datos que cambian frecuentemente debido a las siguientes razones:

1. **Costo**: Fine-tuning GPT-4 puede costar USD 50K a 2M, mientras que RAG utiliza el modelo base sin cambios.
2. **Velocidad**: RAG incorpora nuevos documentos en minutos, lo que la hace más rápida y eficiente

---
## Celda 8 — Inspección del estado de la base de datos

Útil para verificar cuántos chunks fueron indexados y con qué longitud promedio.


In [8]:
def db_stats():
    """Muestra un resumen de los chunks indexados por fuente."""
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT
                    source,
                    COUNT(*)                  AS total_chunks,
                    AVG(LENGTH(content))::INT AS avg_chars
                FROM document_chunks
                GROUP BY source
                ORDER BY source;
            """)
            rows = cur.fetchall()

    print(f"{'Fuente':<35} {'Chunks':>8} {'Avg chars':>10}")
    print("-" * 55)
    for r in rows:
        print(f"{r[0]:<35} {r[1]:>8} {r[2]:>10}")


db_stats()


Fuente                                Chunks  Avg chars
-------------------------------------------------------
techpulse_ia_2025.pdf                      6       1135


---
## Celda 9 — Tu turno

Probá tu propia pregunta sobre el documento indexado.
También podés indexar otro PDF y comparar los resultados.


In [23]:
# Pregunta libre
mi_pregunta = "¿Cuáles son las principales amenazas de seguridad en sistemas RAG?"

print(f"Pregunta: {mi_pregunta}\n")
print(ask(mi_pregunta))


Pregunta: ¿Cuáles son las principales amenazas de seguridad en sistemas RAG?

La pregunta se refiere a las principales amenazas de seguridad en sistemas RAG (Retrieval-Augmented Generation). Según el texto proporcionado, algunas de las amenazas mencionadas incluyen:

* Alucinaciones: LLM mezcla contexto recuperado con información inventada
* Prompt injection en sistemas RAG con inputs no confiables
* Filtración de información confidencial a través de respuestas
* Dependencia de APIs externas y riesgo de discontinuación

Estas amenazas se mencionan como principales preocupaciones de seguridad para los sistemas RAG, lo que sugiere que la seguridad es un aspecto crítico en el desarrollo y implementación de estos sistemas.
